In [1]:
import requests
from langchain_ollama import ChatOllama

# Ollama typically runs on localhost:11434
ollama_url = "http://localhost:11434"

try:
    # Send a request to check if the local Ollama server is active
    response = requests.get(ollama_url)
    if response.status_code == 200:
        print("Ollama server is running locally!")
        
        # Initialize the model (replace 'llama3' with your downloaded model)
        llm = ChatOllama(model="qwen2.5:3b", base_url=ollama_url)
        print("ChatOllama initialized successfully.")
    else:
        print(f"Ollama server responded with status code: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("Ollama server is not running. Please start the Ollama application.")


Ollama server is running locally!
ChatOllama initialized successfully.


In [2]:
from langchain_ollama import ChatOllama

llm_ollama= ChatOllama(model='qwen2.5:3b', temperature=0)

In [3]:
# from openai.types import model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# it only give the content part of the output (task3)
from langchain_core.output_parsers import StrOutputParser

llm_ollama= ChatOllama(model='qwen2.5:3b', temperature=0)
my_messages = [
    SystemMessage(content="you are a gen-z assistant who always answer in fun way"),
    HumanMessage(content="hey")
]

llm_ollama.invoke(my_messages).content

'Hey there! It\'s like saying "hello" but with extra enthusiasm and a big smile. How’s your day so far? Ready for some fun?'

# **Chain with Parallerl Chains**

In [4]:
# prompt 1st TASK

prompt_template = ChatPromptTemplate([
    ('system','You are a movie summarizer'),
    ('human','Please summarize the movie in brief : {input}')
])

In [5]:
# TASK 2 LLM

llm_ollama= ChatOllama(model='qwen2.5:3b', temperature=0)


In [6]:
# TASK 3 Parsers

parcer = StrOutputParser()

In [7]:
# TASK 4 custom function

from langchain_core.runnables import RunnableLambda

def dict_maker(text: str)-> dict :
    return {"text": text}

dict_maker_runnable= RunnableLambda(dict_maker)

## **Parallel Chain 1**


In [8]:
# TASK 1 PROMPT

LinkedIn_post = ChatPromptTemplate.from_messages([
    ('system', 'You are a linkedIn post generator'),
    ('human', 'Create a post for the following text for LinkedIn : {text}')
])

# TASK 2 LLM

llm_ollama= ChatOllama(model='qwen2.5:3b', temperature=0)

# TASK 3 Str PARSER

parcer = StrOutputParser()

chain_linkedIn = LinkedIn_post | llm_ollama | parcer

## **Parallel Chain 2**

In [9]:
from langchain_core.runnables import RunnableParallel

In [12]:
def insta_chain(text : dict):

    text = text['text']
    # TASK 1 PROMPT

    Insta_prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are a Instagram post generator'),
        ('human', 'Create a post for the following text for Instagram : {text}')
    ])

    # TASK 2 LLM

    llm_ollama= ChatOllama(model='qwen2.5:3b', temperature=0)

    # TASK 3 Str PARSER

    parcer = StrOutputParser()

    chain_insta = Insta_prompt | llm_ollama | parcer

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

## **Final Orchestration**

In [13]:
final_chain = (
    prompt_template |
    llm_ollama |
    parcer |
    dict_maker_runnable |
    RunnableParallel(branches={ 'linkedIn': chain_linkedIn, 'Instagram': insta_chain_runnable})
)

In [14]:
final_chain.invoke('iron man')

{'branches': {'linkedIn': "Here’s a LinkedIn post based on the provided text:\n\n---\n\n🚀 **Iron Man: A Journey from Playboy to Hero!** 🚀\n\nMarvel's Iron Man is an action-packed superhero film that delves into the life of Tony Stark, a wealthy industrialist and genius inventor. When captured during a conflict, Stark leverages his advanced military technology to escape and builds a high-tech suit of armor for both self-defense and justice.\n\nThe movie beautifully showcases Stark’s transformation from a self-centered playboy to a reluctant hero who fights for what is right using his Iron Man suit. It's packed with intense action sequences, witty dialogue, and thought-provoking themes like responsibility and the consequences of one's actions.\n\nIron Man not only captivates audiences but also introduces Tony Stark/Iron Man as a key member of the Avengers team in future installments. #IronMan #SuperheroMovies #Avengers\n\n---\n\nFeel free to adjust any details or add your own personal to

## **Chain as a runnable**

In [17]:
# TASK 1 BUFITY FUNCTION

def beautify(final_response: dict)->dict :
    linkedIn_response= final_response['branches']['linkedIn']
    instagram_response= final_response['branches']['Instagram']

    return {'LINKEDIN': linkedIn_response, 'INSTAGRAM': instagram_response}

beautify_runnable = RunnableLambda(beautify)

# TASK 2 FINAL CHAIN

# final_chain the product of a chain is a runnable you dont need to make it runnable as to a function

# beaytified chain

beautify_chain = final_chain | beautify_runnable 

beautify_chain.invoke('avengers:endgame')

{'LINKEDIN': 'Here’s a LinkedIn post based on the provided text:\n\n---\n\n🌟 **Avengers: Endgame - A Culmination of Epic Adventures** 🌟\n\n"Avengers: Endgame," an epic superhero film, brings together all the Avengers characters to save the world from Thanos, who has wiped out half of all life in the universe. The story follows Tony Stark (Iron Man), Steve Rogers (Captain America), and others as they gather to reverse Thanos\' actions by obtaining the Infinity Stones and using them to restore six of the stones that were previously destroyed.\n\nThis includes retrieving the Tesseract from where it was hidden, leading to a series of battles against various threats including Loki, Ronin, and other villains. The climax involves Tony Stark sacrificing himself to activate the remaining Infinity Stone, allowing for the resurrection of many fallen Avengers and the restoration of life in half of the universe.\n\nAvengers: Endgame is not just an action-packed film; it\'s a culmination of the Marv